<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed, test_size=0.2):
    X, y = fetch_openml(
        "Fashion-MNIST",
        version=1,
        as_frame=False,
        return_X_y=True,
        data_home="./.sklearn_data",
    )
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=seed,
    )
    return X_train, X_test, y_train, y_test


# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

def train_random_forest(X_train, y_train, seed):
    model = RandomForestClassifier(
        n_estimators=100,
        random_state=seed,
        n_jobs=-1,
    )
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed):
    base_tree = DecisionTreeClassifier(max_depth=1, random_state=seed)
    model = AdaBoostClassifier(
        estimator=base_tree,
        n_estimators=100,
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model


# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)


Nao e estritamente necessario normalizar os dados para `RandomForestClassifier` e `AdaBoostClassifier` com arvores como estimador base, porque esses modelos sao baseados em regras de divisao e nao em distancia ou escala das features.

Mesmo assim, manter os pixels em uma escala consistente pode ajudar na organizacao do experimento. Neste caso, a ausencia de normalizacao nao compromete a corretude metodologica do pipeline.


# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [ ]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc


Observacao: nesta atividade, ensemble learning substitui a analise de profundidade unica da arvore. O foco aqui passa a ser comparar desempenho, reproducibilidade e sensibilidade a hiperparametros entre `RandomForest` e `AdaBoost`.


# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def compute_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="macro"),
        "recall": recall_score(y_test, y_pred, average="macro"),
        "f1": f1_score(y_test, y_pred, average="macro"),
    }

X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)
ab_model = train_adaboost(X_train, y_train, seed=42)

rf_metrics = compute_metrics(rf_model, X_test, y_test)
ab_metrics = compute_metrics(ab_model, X_test, y_test)

print("Random Forest:", rf_metrics)
print("AdaBoost:", ab_metrics)


Em geral, o `Random Forest` tende a apresentar melhor desempenho inicial no Fashion MNIST, com acuracia e F1-score superiores ao `AdaBoost` usando stump como estimador base.

Isso acontece porque o `Random Forest` combina varias arvores mais expressivas e costuma capturar melhor a complexidade das imagens. Ja o `AdaBoost` com arvores rasas pode ter mais dificuldade para modelar fronteiras de decisao complexas.


**Solução**:

In [ ]:
for seed in [42, 7]:
    rf_acc = run_pipeline(model_type="rf", seed=seed)
    ab_acc = run_pipeline(model_type="ab", seed=seed)
    print(f"Seed = {seed}")
    print(f"  Random Forest acc: {rf_acc:.4f}")
    print(f"  AdaBoost acc: {ab_acc:.4f}")


Os resultados podem mudar ligeiramente quando a seed muda, porque a divisao treino/teste e o processo interno dos modelos dependem da aleatoriedade controlada por `random_state`.

Ainda assim, o experimento continua sendo reprodutivel, porque a mesma seed gera exatamente a mesma divisao e o mesmo treinamento. Ou seja, ha variacao entre seeds diferentes, mas consistencia quando a configuracao e repetida.


In [ ]:
X_train, X_test, y_train, y_test = load_data(seed=42)

rf_model = train_random_forest(X_train, y_train, seed=42)

train_acc_rf = evaluate(rf_model, X_train, y_train)
test_acc_rf = evaluate(rf_model, X_test, y_test)

print(f"Random Forest - treino: {train_acc_rf:.4f}")
print(f"Random Forest - teste: {test_acc_rf:.4f}")


Se a acuracia de treino ficar bem acima da acuracia de teste, existe indicio de overfitting. Isso e relativamente comum em `RandomForest`, especialmente com muitas arvores profundas e sem restricoes adicionais.

Entre os dois modelos, o `Random Forest` tende a sofrer mais overfitting quando muito flexivel. O `AdaBoost` tambem pode superajustar, mas com stump (`max_depth=1`) ele costuma ser mais restrito.


In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

X_train, X_test, y_train, y_test = load_data(seed=42)

print("Random Forest")
for n_estimators in [50, 100, 200]:
    rf_model = RandomForestClassifier(
        n_estimators=n_estimators,
        random_state=42,
        n_jobs=-1,
    )
    rf_model.fit(X_train, y_train)
    rf_acc = evaluate(rf_model, X_test, y_test)
    print(f"  n_estimators={n_estimators}: {rf_acc:.4f}")

print("\nAdaBoost")
for n_estimators in [50, 100, 200]:
    ab_model = AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=n_estimators,
        random_state=42,
    )
    ab_model.fit(X_train, y_train)
    ab_acc = evaluate(ab_model, X_test, y_test)
    print(f"  n_estimators={n_estimators}: {ab_acc:.4f}")


O desempenho normalmente melhora ate certo ponto com o aumento de `n_estimators`, mas o ganho tende a diminuir conforme o ensemble cresce.

Na pratica, o `AdaBoost` costuma ser mais sensivel a esse hiperparametro, porque cada novo estimador corrige erros dos anteriores de forma sequencial. Ja o `Random Forest` tende a variar de forma mais estavel.


In [ ]:
1. A acuracia nao e suficiente sozinha, porque pode esconder desequilibrios entre classes e nao mostrar onde o modelo esta errando. Por isso, precisao, recall e F1-score ajudam a formar uma avaliacao mais completa.

2. Para reduzir a chance de um resultado ocorrer por acaso, usamos seeds controladas, divisao estratificada e repeticao do experimento com seeds diferentes. Isso melhora a confiabilidade e permite verificar estabilidade.

3. Dois problemas metodologicos possiveis sao: avaliar apenas uma divisao treino/teste e nao explorar validacao cruzada; comparar modelos sem investigar tuning de hiperparametros de forma equilibrada. Ambos podem distorcer a conclusao.

4. O pipeline e confiavel como ponto de partida, porque organiza carregamento, treino e avaliacao de forma reproduzivel. Ainda assim, ele pode ser fortalecido com validacao cruzada, busca sistematica de hiperparametros e analise de matriz de confusao.
